# 🎵 YuE2 Studio — Kaggle Dual Tesla T4 (2x 16GB VRAM) Server

[![Kaggle](https://img.shields.io/badge/Kaggle-GPU%20T4%20x%202-blue?logo=kaggle)](https://www.kaggle.com)
[![YuE2 Studio](https://img.shields.io/badge/GitHub-YuE2--Studio-pink?logo=github)](https://github.com/timoncool/YuE2-Studio)
[![Total VRAM](https://img.shields.io/badge/VRAM-32%20GB%20(2x%2016GB)-success?logo=nvidia)](https://www.kaggle.com)

Этот блокнот разворачивает облачный бэкенд **YuE2 Studio** на **Kaggle с двумя видеокартами Tesla T4 (32 ГБ суммарной VRAM)**.

### ⚡ Преимущества Kaggle 2x T4 перед Colab:
- **30 часов GPU в неделю бесплатно** — лимиты значительно выше и сессии стабильнее.
- **Две независимые видеокарты Tesla T4**:
  - **GPU 0 (16 ГБ VRAM):** Выделена на 100% под языковую модель генерации музыки **YuE2 (Q8_0 или даже полновесный BF16)**.
  - **GPU 1 (16 ГБ VRAM):** Выделена на 100% под транскрибацию **Аудио в MIDI (MuScriptor Large 1.4B)**.
- **Ноль конфликтов памяти (No CUDA OOM):** Обе тяжелые нейросети живут в памяти одновременно без вылетов.
- **Скоростной интернет Kaggle:** Загрузка весов из Hugging Face на скорости до 100–150 МБ/с (3.6 ГБ скачиваются за 30 секунд).

---

### 🛠️ ВАЖНАЯ НАСТРОЙКА KAGGLE ПЕРЕД ЗАПУСКОМ:
В правой боковой панели блокнота Kaggle проверьте переключатели в блоке **Settings**:
1. **Accelerator:** выберите **`GPU T4 x 2`** *(не T4 x 1, а именно T4 x 2)*.
2. **Internet:** включите **`Internet On`** *(обязательно для туннеля и загрузки моделей)*.


### 1️⃣ Шаг 1: Проверка окружения и наличия 2x Tesla T4
Запустите ячейку ниже, чтобы убедиться, что Kaggle выделил обе видеокарты.

In [ ]:
!nvidia-smi

import torch
gpu_count = torch.cuda.device_count()
print(f"\n🎮 Доступно GPU: {gpu_count}")
for i in range(gpu_count):
    print(f"  • GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / (1024**3):.1f} GB VRAM)")

if gpu_count < 2:
    print("\n⚠️ ВНИМАНИЕ: Назначена только 1 GPU. В правой панели Kaggle (Settings -> Accelerator) выберите 'GPU T4 x 2'!")
else:
    print("\n🎉 Отлично! Обе видеокарты Tesla T4 готовы к параллельной работе.")


### 🧹 Шаг 1.4 (При необходимости): Очистка диска и проверка готовых моделей

На Kaggle действует строгий лимит рабочего диска — **19.5 ГБ**.
Если при скачивании диск заполнился (`19.5GiB / 19.5GiB` или `Errno 28 No space left on device`):
1. Запустите эту ячейку — она **удалит тяжёлую папку `models_bf16` (~9-10 ГБ)**, которая не нужна для запуска квантованной модели `Q8_0`.
2. Очистит временные кэши и переместит скачанные модели из `YuE2/models/` в рабочую папку `models/`.
3. Покажет свободное место и список готовых моделей.

In [ ]:
# @title 🧹 Очистить диск и проверить готовые модели { run: "auto" }
REMOVE_HEAVY_BF16 = True  # @param {type:"boolean"}
CLEAR_CACHE = True  # @param {type:"boolean"}

import os
import shutil
from pathlib import Path

work_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
models_dir = work_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

print("🚀 Анализ и очистка рабочего диска...")

# 1. Удаление тяжелых исходников BF16 (освобождает ~9-10 ГБ)
if REMOVE_HEAVY_BF16:
    bf16_targets = [
        work_dir / "models_bf16",
        work_dir / "YuE2" / "models_bf16",
    ]
    for p in list(work_dir.rglob("models_bf16")):
        bf16_targets.append(p)
    for p in list(work_dir.rglob("*BF16*.gguf")):
        bf16_targets.append(p)

    for target in set(bf16_targets):
        if target.exists():
            try:
                if target.is_dir():
                    shutil.rmtree(target, ignore_errors=True)
                else:
                    target.unlink(missing_ok=True)
                print(f"  🗑️ Удалено: {target.name} (освобождено место)")
            except Exception as exc:
                print(f"  ⚠️ Ошибка при удалении {target}: {exc}")

# 2. Очистка кэшей
if CLEAR_CACHE:
    cache_targets = [
        Path("/root/.cache/pip"),
        Path("/root/.cache/torch"),
        Path("/root/.cache/huggingface"),
        Path("/tmp/gdown"),
    ]
    for c in cache_targets:
        if c.exists():
            shutil.rmtree(c, ignore_errors=True)

# 3. Перенос полезных файлов из подпапок YuE2 в models_dir и work_dir
for gdir in [work_dir / "YuE2", work_dir / "YuE2" / "models"]:
    if gdir.exists() and gdir != models_dir:
        for f in gdir.iterdir():
            if f.is_file():
                if f.name.endswith(".gguf") or f.name.endswith(".safetensors"):
                    dest = models_dir / f.name
                    if not dest.exists():
                        shutil.move(str(f), str(dest))
                        print(f"  📦 Перемещен в models: {f.name}")
                elif f.name == "yue-server" or ".so" in f.name:
                    dest = work_dir / f.name
                    if not dest.exists():
                        shutil.move(str(f), str(dest))
                        dest.chmod(0o755)
                        print(f"  ⚙️ Перемещен бинарник движка: {f.name}")

# 4. Проверка свободного места
total, used, free = shutil.disk_usage(str(work_dir))
free_gb = free / (1024 ** 3)
total_gb = total / (1024 ** 3)
used_gb = used / (1024 ** 3)

print("\n" + "=" * 60)
print(f"💾 Диск /kaggle/working: СВОБОДНО {free_gb:.2f} ГБ из {total_gb:.2f} ГБ (занято {used_gb:.2f} ГБ)")
print("=" * 60)

# 5. Список готовых моделей
print(f"\n📂 Готовые файлы в {models_dir}:")
found_models = list(models_dir.glob("*"))
if found_models:
    for mf in sorted(found_models):
        if mf.is_file():
            size_mb = mf.stat().st_size / (1024 * 1024)
            print(f"  ✓ {mf.name} ({size_mb:.1f} MB)")
else:
    print("  (пока пусто)")

bin_target = work_dir / "yue-server"
if bin_target.exists():
    print(f"  ✓ yue-server ({bin_target.stat().st_size / (1024*1024):.1f} MB) — движок готов!")

has_bb = any("Q8_0" in f.name for f in found_models)
has_vae = any("Vae" in f.name for f in found_models)
has_lora = any("joint_v9" in f.name for f in found_models)
if has_bb and has_vae and has_lora:
    print("\n🎉 Отлично! Все основные компоненты YuE2 (Q8_0, VAE, LoRA) уже на месте.")
    print("👉 Можно сразу запускать Шаг 2 (Запуск сервера)!")
elif free_gb > 5.0:
    print(f"\n💡 Свободного места достаточно ({free_gb:.1f} ГБ). Можно скачивать модели в Шаге 1.5.")


### 📁 Шаг 1.5 (Опционально): Умное скачивание с Google Диска (выборочно)

Чтобы не забивать диск 19.5 ГБ лишними файлами, эта ячейка умеет качать **выборочно**:
- **Режим Q8_0:** Скачивает только рабочие модели (~4.5 ГБ) и автоматически пропускает тяжёлый `models_bf16`.
- **Ссылка на файл:** Если вставить ссылку на конкретный `.gguf` или `.safetensors`, скачается только он.
- **Ссылка на папку:** Сканирует папку без предварительной загрузки и берёт только необходимые компоненты.

*(Если ничего не вводить — при запуске Шага 2 модели автоматически скачаются из Hugging Face за ~30 секунд)*.

In [ ]:
# @title 📁 Скачать модели с Google Диска через gdown (выборочно) { run: "auto" }
GDRIVE_URL = ""  # @param {type:"string"}
DOWNLOAD_MODE = "Q8_0 (Рекомендуется: пропустить BF16, экономить диск)"  # @param ["Q8_0 (Рекомендуется: пропустить BF16, экономить диск)", "Только указанный файл (если ссылка на файл)", "Вся папка целиком (Осторожно: лимит диска 19.5 ГБ)"]

import os
import re
import shutil
import subprocess
from pathlib import Path

work_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
models_dir = work_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

if not GDRIVE_URL.strip():
    print("ℹ️ Ссылка не указана. Модели будут автоматически загружены из Hugging Face при запуске Шага 2.")
else:
    url = GDRIVE_URL.strip()
    if shutil.which("gdown") is None:
        print("📦 Установка gdown...")
        subprocess.check_call(["pip", "install", "-q", "gdown"])

    import gdown
    from gdown.parse_url import parse_url

    file_id, _ = parse_url(url)
    folder_match = re.search(r"folders/([a-zA-Z0-9_-]+)", url)
    folder_id = folder_match.group(1) if folder_match else None

    # Вариант 1: Прямая ссылка на файл
    if file_id and (not folder_id or "file" in url or "Только указанный файл" in DOWNLOAD_MODE):
        print(f"⏳ Скачивание файла с Google Диска (ID: {file_id})...")
        try:
            downloaded = gdown.download(id=file_id, output=str(models_dir), quiet=False, resume=True)
            if downloaded:
                p = Path(downloaded)
                if p.name == "yue-server" or ".so" in p.name:
                    shutil.move(str(p), str(work_dir / p.name))
                    (work_dir / p.name).chmod(0o755)
                print(f"✅ Файл успешно сохранён: {Path(downloaded).name}")
        except Exception as exc:
            print(f"⚠️ Ошибка скачивания файла: {exc}")

    # Вариант 2: Ссылка на папку
    elif folder_id:
        print(f"⏳ Сканирование папки Google Диска (ID: {folder_id})...")
        is_q8_profile = "Q8_0" in DOWNLOAD_MODE
        try:
            items = gdown.download_folder(
                url=f"https://drive.google.com/drive/folders/{folder_id}",
                output=str(work_dir),
                skip_download=True,
            )
            print(f"📋 Найдено объектов в папке Google Диска: {len(items)}")

            download_queue = []
            for it in items:
                p_lower = str(it.path).lower()
                if is_q8_profile and ("bf16" in p_lower or "models_bf16" in p_lower or "fp16" in p_lower):
                    print(f"  ⏭️ Пропущен (BF16 не нужен для Q8_0): {it.path}")
                    continue
                download_queue.append(it)

            print(f"\n🚀 К скачиванию: {len(download_queue)} нужных файлов:")
            for it in download_queue:
                print(f"  • {it.path}")

            for it in download_queue:
                # Проверяем свободное место перед каждым файлом
                _, _, cur_free = shutil.disk_usage(str(work_dir))
                if cur_free < 2 * 1024**3:
                    print("⚠️ ВНИМАНИЕ: Свободного места осталось менее 2 ГБ! Остановка загрузки.")
                    break
                print(f"\n📥 Скачивание: {it.path}...")
                dest = work_dir / it.path
                dest.parent.mkdir(parents=True, exist_ok=True)
                gdown.download(id=it.id, output=str(dest), quiet=False, resume=True)

            print("\n✅ Загрузка завершена!")
        except Exception as exc:
            print(f"⚠️ Ошибка выборочного скачивания: {exc}")
            print("🔄 Запуск резервного скачивания через консольную команду...")
            try:
                subprocess.run([
                    "gdown", "--folder", f"https://drive.google.com/drive/folders/{folder_id}",
                    "-O", str(work_dir), "--remaining-ok"
                ], check=True)
            except Exception as e2:
                print(f"⚠️ Ошибка резервного скачивания: {e2}")

        # Организуем файлы по папкам
        for gdir in [work_dir / "YuE2", work_dir / "YuE2" / "models"]:
            if gdir.exists() and gdir != models_dir:
                for f in gdir.iterdir():
                    if f.is_file():
                        if f.name.endswith(".gguf") or f.name.endswith(".safetensors"):
                            dest = models_dir / f.name
                            if not dest.exists():
                                shutil.move(str(f), str(dest))
                        elif f.name == "yue-server" or ".so" in f.name:
                            dest = work_dir / f.name
                            if not dest.exists():
                                shutil.move(str(f), str(dest))
                                dest.chmod(0o755)

        if is_q8_profile:
            shutil.rmtree(work_dir / "models_bf16", ignore_errors=True)
            shutil.rmtree(work_dir / "YuE2" / "models_bf16", ignore_errors=True)

    # Итоговый отчет о диске
    total, used, free = shutil.disk_usage(str(work_dir))
    print("\n" + "=" * 60)
    print(f"💾 Свободно на диске: {free / (1024**3):.2f} ГБ из {total / (1024**3):.2f} ГБ")
    print("📂 Готовые файлы в", models_dir)
    for mf in sorted(models_dir.glob("*")):
        if mf.is_file():
            print(f"  ✓ {mf.name} ({mf.stat().st_size / (1024*1024):.1f} MB)")
    print("=" * 60)


### 2️⃣ Шаг 2: Запуск сервера YuE2 Studio
Выберите квантование (`Q8_0` рекомендуется) и тип туннеля (`cloudflare` без регистрации или `ngrok`).
Сервер автоматически скомпилирует ядро `yue2.cpp` под архитектуру Turing (sm_75), распределит модели по двум GPU и выдаст публичную ссылку.

In [ ]:
from __future__ import annotations

# @title ⚙️ Параметры и запуск сервера (Kaggle Dual-T4) { run: "auto" }
QUANT = "Q8_0"  # @param ["Q8_0", "BF16", "Q6_K", "Q5_K_M"]
TUNNEL = "cloudflare"  # @param ["cloudflare", "ngrok"]
NGROK_TOKEN = ""  # @param {type:"string"}
PORT = 18087  # @param {type:"integer"}


import argparse
import http.server
import json
import logging
import os
import re
import shutil
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request
from dataclasses import dataclass
from pathlib import Path
from typing import Final

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
logger: logging.Logger = logging.getLogger("yue2-kaggle-2xt4")

HF_REPO_YUE_GGUF: Final[str] = "Serveurperso/YuE2-GGUF"
HF_REV_YUE_GGUF: Final[str] = "64b030e3deb6e8150d2b7c0db641ef5a17eca8a3"
HF_REPO_COMPANION: Final[str] = (
    "Mothersuperior/yue2-mothersuperior-realaudio-tokenizer-v4"
)
HF_REV_COMPANION: Final[str] = "e2e63d859f3af879baf1b4d4e9f22d1eeda6fde5"

DEFAULT_PORT: Final[int] = 18087
CLOUDFLARED_URL: Final[str] = (
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
)


@dataclass(frozen=True)
class ModelFileConfig:
    """Download specification for a single model weight file."""

    filename: str
    url: str
    size_mb: int
    role: str


MODEL_CATALOG: Final[dict[str, list[ModelFileConfig]]] = {
    "q8_0": [
        ModelFileConfig(
            filename="YuE2-3B-Q8_0.gguf",
            url=f"https://huggingface.co/{HF_REPO_YUE_GGUF}/resolve/{HF_REV_YUE_GGUF}/YuE2-3B-Q8_0.gguf",
            size_mb=3550,
            role="backbone",
        ),
        ModelFileConfig(
            filename="YuE2-Vae-F32.gguf",
            url=f"https://huggingface.co/{HF_REPO_YUE_GGUF}/resolve/{HF_REV_YUE_GGUF}/YuE2-Vae-F32.gguf",
            size_mb=506,
            role="vae",
        ),
        ModelFileConfig(
            filename="nar_lora_joint_v9.safetensors",
            url=f"https://huggingface.co/{HF_REPO_COMPANION}/resolve/{HF_REV_COMPANION}/nar_lora_joint_v9.safetensors",
            size_mb=134,
            role="companion",
        ),
        ModelFileConfig(
            filename="SheetSage2-Q8_0.gguf",
            url=f"https://huggingface.co/{HF_REPO_YUE_GGUF}/resolve/{HF_REV_YUE_GGUF}/SheetSage2-Q8_0.gguf",
            size_mb=913,
            role="transcriber",
        ),
    ],
    "bf16": [
        ModelFileConfig(
            filename="YuE2-3B-BF16.gguf",
            url=f"https://huggingface.co/{HF_REPO_YUE_GGUF}/resolve/{HF_REV_YUE_GGUF}/YuE2-3B-BF16.gguf",
            size_mb=6670,
            role="backbone",
        ),
        ModelFileConfig(
            filename="YuE2-Vae-F32.gguf",
            url=f"https://huggingface.co/{HF_REPO_YUE_GGUF}/resolve/{HF_REV_YUE_GGUF}/YuE2-Vae-F32.gguf",
            size_mb=506,
            role="vae",
        ),
        ModelFileConfig(
            filename="nar_lora_joint_v9.safetensors",
            url=f"https://huggingface.co/{HF_REPO_COMPANION}/resolve/{HF_REV_COMPANION}/nar_lora_joint_v9.safetensors",
            size_mb=134,
            role="companion",
        ),
        ModelFileConfig(
            filename="SheetSage2-F32.gguf",
            url=f"https://huggingface.co/{HF_REPO_YUE_GGUF}/resolve/{HF_REV_YUE_GGUF}/SheetSage2-F32.gguf",
            size_mb=2520,
            role="transcriber",
        ),
    ],
    "q6_k": [
        ModelFileConfig(
            filename="YuE2-3B-Q6_K.gguf",
            url=f"https://huggingface.co/{HF_REPO_YUE_GGUF}/resolve/{HF_REV_YUE_GGUF}/YuE2-3B-Q6_K.gguf",
            size_mb=2820,
            role="backbone",
        ),
        ModelFileConfig(
            filename="YuE2-Vae-F32.gguf",
            url=f"https://huggingface.co/{HF_REPO_YUE_GGUF}/resolve/{HF_REV_YUE_GGUF}/YuE2-Vae-F32.gguf",
            size_mb=506,
            role="vae",
        ),
        ModelFileConfig(
            filename="nar_lora_joint_v9.safetensors",
            url=f"https://huggingface.co/{HF_REPO_COMPANION}/resolve/{HF_REV_COMPANION}/nar_lora_joint_v9.safetensors",
            size_mb=134,
            role="companion",
        ),
        ModelFileConfig(
            filename="SheetSage2-Q6_K.gguf",
            url=f"https://huggingface.co/{HF_REPO_YUE_GGUF}/resolve/{HF_REV_YUE_GGUF}/SheetSage2-Q6_K.gguf",
            size_mb=726,
            role="transcriber",
        ),
    ],
    "q5_k_m": [
        ModelFileConfig(
            filename="YuE2-3B-Q5_K_M.gguf",
            url=f"https://huggingface.co/{HF_REPO_YUE_GGUF}/resolve/{HF_REV_YUE_GGUF}/YuE2-3B-Q5_K_M.gguf",
            size_mb=2480,
            role="backbone",
        ),
        ModelFileConfig(
            filename="YuE2-Vae-F32.gguf",
            url=f"https://huggingface.co/{HF_REPO_YUE_GGUF}/resolve/{HF_REV_YUE_GGUF}/YuE2-Vae-F32.gguf",
            size_mb=506,
            role="vae",
        ),
        ModelFileConfig(
            filename="nar_lora_joint_v9.safetensors",
            url=f"https://huggingface.co/{HF_REPO_COMPANION}/resolve/{HF_REV_COMPANION}/nar_lora_joint_v9.safetensors",
            size_mb=134,
            role="companion",
        ),
        ModelFileConfig(
            filename="SheetSage2-Q5_K_M.gguf",
            url=f"https://huggingface.co/{HF_REPO_YUE_GGUF}/resolve/{HF_REV_YUE_GGUF}/SheetSage2-Q5_K_M.gguf",
            size_mb=640,
            role="transcriber",
        ),
    ],
}


def detect_environment() -> tuple[Path, int]:
    """Detects working directory and available GPU count (Kaggle or Colab/Linux)."""
    kaggle_dir = Path("/kaggle/working")
    work_dir = kaggle_dir if kaggle_dir.exists() else Path("/content") if Path("/content").exists() else Path.cwd()

    gpu_count = 0
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            text=True,
        )
        lines = [line.strip() for line in out.strip().splitlines() if line.strip()]
        gpu_count = len(lines)
        logger.info("Detected %d NVIDIA GPU(s):", gpu_count)
        for i, line in enumerate(lines):
            logger.info("  [GPU %d] %s", i, line)
    except Exception as exc:  # noqa: BLE001
        logger.warning("Could not query GPUs via nvidia-smi: %s", exc)

    return work_dir, gpu_count


def ensure_system_packages() -> None:
    """Installs required system packages for building yue2.cpp and running aria2c."""
    packages = ["cmake", "build-essential", "aria2", "ninja-build", "git"]
    missing: list[str] = []
    for pkg in packages:
        check_binary = "aria2c" if pkg == "aria2" else pkg
        check_binary = "gcc" if pkg == "build-essential" else check_binary
        check_binary = "ninja" if pkg == "ninja-build" else check_binary
        if shutil.which(check_binary) is None:
            missing.append(pkg)

    if missing and shutil.which("apt-get"):
        logger.info("Installing missing system packages: %s", ", ".join(missing))
        try:
            subprocess.check_call(["apt-get", "update", "-qq"])
            subprocess.check_call(
                ["apt-get", "install", "-y", "-qq", "--no-install-recommends"] + missing
            )
        except (subprocess.CalledProcessError, OSError) as exc:
            logger.warning("Package installation notice: %s", exc)

    if shutil.which("muscriptor") is None:
        logger.info("Installing muscriptor for GPU Audio-to-MIDI transcription...")
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "muscriptor"]
            )
        except (subprocess.CalledProcessError, OSError) as exc:
            logger.warning("Muscriptor pip notice: %s", exc)


def is_binary_runnable(binary: Path) -> bool:
    """Verifies that a compiled binary can load its dynamic libraries without error 127."""
    if not binary.exists():
        return False
    work_dir = binary.parent
    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = f"{work_dir}:{env.get('LD_LIBRARY_PATH', '')}"
    try:
        res = subprocess.run(
            [str(binary), "--help"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=5,
            env=env,
        )
        if res.returncode == 127:
            err = res.stderr.decode(errors="ignore").strip()
            logger.warning("Binary %s is incompatible with current CUDA (error 127: %s)", binary.name, err)
            return False
        return True
    except Exception as exc:
        logger.warning("Binary check notice: %s", exc)
        return False


def build_yue_engine(work_dir: Path) -> Path:
    """Builds yue2.cpp with CUDA sm_75 architecture for Tesla T4."""
    local_target = work_dir / "yue-server"
    local_libs = list(work_dir.glob("lib*.so*"))

    if local_target.exists() and local_libs and is_binary_runnable(local_target):
        local_target.chmod(0o755)
        logger.info("Found cached working yue-server with libraries at %s", local_target)
        return local_target

    # Check if gdown placed yue-server in work_dir / "YuE2"
    gdown_dir = work_dir / "YuE2"
    if gdown_dir.exists():
        gdown_target = gdown_dir / "yue-server"
        gdown_libs = list(gdown_dir.glob("lib*.so*"))
        if gdown_target.exists() and gdown_libs:
            shutil.copy2(gdown_target, local_target)
            local_target.chmod(0o755)
            for lib in gdown_libs:
                shutil.copy2(lib, work_dir / lib.name)
            if is_binary_runnable(local_target):
                logger.info("Restored compatible yue-server from Google Drive (%s)...", gdown_dir)
                return local_target
            logger.warning("yue-server from Google Drive has mismatched CUDA libraries. Rebuilding natively on Kaggle...")
            local_target.unlink(missing_ok=True)

    # Check /kaggle/input for pre-compiled yue-server dataset
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for candidate in kaggle_input.rglob("yue-server"):
            if candidate.is_file():
                cand_dir = candidate.parent
                cand_libs = list(cand_dir.glob("lib*.so*"))
                if cand_libs:
                    shutil.copy2(candidate, local_target)
                    local_target.chmod(0o755)
                    for lib in cand_libs:
                        shutil.copy2(lib, work_dir / lib.name)
                    if is_binary_runnable(local_target):
                        logger.info("Restoring pre-compiled yue-server from Kaggle Input: %s", candidate)
                        return local_target
                    logger.warning("yue-server from Kaggle Input is not runnable on this system. Rebuilding...")
                    local_target.unlink(missing_ok=True)

    repo_url = "https://github.com/timoncool/yue2.cpp.git"
    source_dir = work_dir / "yue2.cpp"
    build_dir = source_dir / "build"

    # Ensure libcuda.so symlink exists in Kaggle container (required for CMake CUDA::cuda_driver target)
    for src in [
        Path("/usr/lib/x86_64-linux-gnu/libcuda.so.1"),
        Path("/usr/local/cuda/lib64/stubs/libcuda.so"),
        Path("/usr/local/cuda/targets/x86_64-linux/lib/stubs/libcuda.so"),
    ]:
        if src.exists():
            for dst in [
                Path("/usr/lib/x86_64-linux-gnu/libcuda.so"),
                Path("/usr/local/cuda/lib64/libcuda.so"),
                Path("/usr/local/cuda/targets/x86_64-linux/lib/libcuda.so"),
            ]:
                if not dst.exists():
                    try:
                        dst.symlink_to(src)
                        logger.info("Created CUDA driver symlink: %s -> %s", dst, src)
                    except Exception:
                        pass
            break

    # Clean prior failed build directory
    if build_dir.exists():
        shutil.rmtree(build_dir, ignore_errors=True)

    if not source_dir.exists():
        logger.info("Cloning yue2.cpp repository...")
        subprocess.check_call(
            ["git", "clone", "--recurse-submodules", "-q", repo_url, str(source_dir)]
        )

    logger.info("Configuring CMake for Tesla T4 (Turing sm_75)...")
    cmake_cmd = [
        "cmake",
        "-B",
        str(build_dir),
        "-S",
        str(source_dir),
        "-DCMAKE_BUILD_TYPE=Release",
        "-DGGML_CUDA=ON",
        "-DGGML_CUDA_NO_VMM=ON",
        "-DCMAKE_CUDA_ARCHITECTURES=75",
        "-DCMAKE_POSITION_INDEPENDENT_CODE=ON",
    ]
    for stub_dir in [
        "/usr/local/cuda/lib64/stubs",
        "/usr/local/cuda/targets/x86_64-linux/lib/stubs",
        "/usr/lib/x86_64-linux-gnu",
    ]:
        if Path(stub_dir).exists():
            cmake_cmd.append(f"-DCMAKE_LIBRARY_PATH={stub_dir}")
            break

    if shutil.which("ninja"):
        cmake_cmd.extend(["-G", "Ninja"])
    subprocess.check_call(cmake_cmd)

    cpu_cores = os.cpu_count() or 4
    logger.info("Compiling yue2.cpp with %d CPU threads...", cpu_cores)
    subprocess.check_call(
        ["cmake", "--build", str(build_dir), "--config", "Release", "-j", str(cpu_cores)]
    )

    built_binaries = list(build_dir.rglob("yue-server"))
    if not built_binaries:
        raise FileNotFoundError("Compilation finished but yue-server was not found.")

    target_exe = built_binaries[0]
    shutil.copy2(target_exe, local_target)
    local_target.chmod(0o755)
    logger.info("Successfully built yue-server: %s", local_target)

    # Collect and copy all shared libraries
    for f in build_dir.rglob("*.so*"):
        if f.is_file() and not f.is_symlink():
            try:
                shutil.copy2(f, work_dir / f.name)
            except Exception:
                pass
        elif f.is_symlink():
            try:
                shutil.copy2(f.resolve(), work_dir / f.name)
            except Exception:
                pass

    return local_target


def download_models(
    models_dir: Path, profile: str, include_transcriber: bool = True
) -> dict[str, Path]:
    """Downloads required GGUF weights into models_dir or finds them in /kaggle/input."""
    models_dir.mkdir(parents=True, exist_ok=True)
    configs = MODEL_CATALOG.get(profile, MODEL_CATALOG["q8_0"])
    has_aria2c = shutil.which("aria2c") is not None
    resolved: dict[str, Path] = {}

    kaggle_input = Path("/kaggle/input")

    for cfg in configs:
        if cfg.role == "transcriber" and not include_transcriber:
            continue

        min_bytes = int(cfg.size_mb * 1024 * 1024 * 0.9)

        # 1. Check if attached as a Kaggle Input Dataset
        if kaggle_input.exists():
            matches = list(kaggle_input.rglob(cfg.filename))
            valid = [m for m in matches if m.is_file() and m.stat().st_size > min_bytes]
            if valid:
                logger.info("Found '%s' in Kaggle input: %s", cfg.filename, valid[0])
                resolved[cfg.role] = valid[0]
                continue

        # 2. Check work_dir / "YuE2" (gdown directory)
        work_base = models_dir.parent
        gdown_candidates = [
            work_base / "YuE2" / cfg.filename,
            work_base / "YuE2" / "models" / cfg.filename,
        ]
        found_gdown = False
        for gc in gdown_candidates:
            if gc.exists() and gc.stat().st_size > min_bytes:
                logger.info("Found '%s' from Google Drive (%s)", cfg.filename, gc)
                resolved[cfg.role] = gc
                found_gdown = True
                break
        if found_gdown:
            continue

        # 3. Check local models dir
        dest = models_dir / cfg.filename
        resolved[cfg.role] = dest
        if dest.exists() and dest.stat().st_size > min_bytes:
            logger.info("Model '%s' already downloaded (%d MB).", cfg.filename, cfg.size_mb)
            continue

        # 3. Download via aria2c
        logger.info("Downloading %s (~%d MB) into %s...", cfg.filename, cfg.size_mb, models_dir)
        if has_aria2c:
            subprocess.check_call(
                [
                    "aria2c",
                    "-c",
                    "-x",
                    "16",
                    "-s",
                    "16",
                    "-k",
                    "1M",
                    "--file-allocation=none",
                    "--dir",
                    str(models_dir),
                    "--out",
                    cfg.filename,
                    cfg.url,
                ]
            )
        else:
            urllib.request.urlretrieve(cfg.url, str(dest))
        logger.info("Downloaded %s successfully.", cfg.filename)

    return resolved


def setup_cloudflare_tunnel(port: int, work_dir: Path) -> tuple[subprocess.Popen[str], str]:
    """Downloads and launches cloudflared tunnel returning the public HTTPS URL."""
    cloudflared_bin = work_dir / "cloudflared"
    if not cloudflared_bin.exists():
        logger.info("Downloading cloudflared binary...")
        urllib.request.urlretrieve(CLOUDFLARED_URL, str(cloudflared_bin))
        cloudflared_bin.chmod(0o755)

    cmd = [
        str(cloudflared_bin),
        "tunnel",
        "--url",
        f"http://127.0.0.1:{port}",
        "--no-autoupdate",
    ]
    logger.info("Launching Cloudflare tunnel on port %d...", port)
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
    public_url = ""
    start_time = time.time()
    while time.time() - start_time < 45:
        if proc.stdout:
            line = proc.stdout.readline()
            if not line and proc.poll() is not None:
                break
            match = url_pattern.search(line)
            if match:
                public_url = match.group(0)
                break
        time.sleep(0.1)

    if not public_url:
        logger.warning("Could not automatically extract Cloudflare URL within 45s.")
        public_url = f"http://127.0.0.1:{port}"

    return proc, public_url


def setup_ngrok_tunnel(
    port: int, authtoken: str | None = None
) -> tuple[subprocess.Popen[str] | None, str]:
    """Configures and runs ngrok tunnel using pyngrok."""
    try:
        from pyngrok import conf, ngrok  # type: ignore[import-untyped]
    except ImportError:
        logger.info("Installing pyngrok...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyngrok"])
        from pyngrok import conf, ngrok  # type: ignore[import-untyped]

    token = authtoken or ""
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient  # type: ignore[import-untyped]
            user_secrets = UserSecretsClient()
            token = user_secrets.get_secret("NGROK_TOKEN") or ""
        except Exception:
            pass

    if token:
        conf.get_default().auth_token = token

    try:
        ngrok.kill()
    except Exception:
        pass

    tunnel = ngrok.connect(port, "http")
    public_url = str(tunnel.public_url)
    if public_url.startswith("http://"):
        public_url = "https://" + public_url[7:]
    logger.info("ngrok tunnel active: %s", public_url)
    return None, public_url


def launch_engine(
    executable: Path,
    models: dict[str, Path],
    adapters_dir: Path | None = None,
    port: int = DEFAULT_PORT,
    gpu_id: int = 0,
) -> subprocess.Popen[str]:
    """Runs the yue-server process on the designated GPU (GPU 0 on Kaggle 2xT4)."""
    cmd = [
        str(executable),
        "--model",
        str(models["backbone"]),
        "--vae",
        str(models["vae"]),
        "--host",
        "0.0.0.0",
        "--port",
        str(port),
    ]
    if "companion" in models and models["companion"].exists():
        cmd.extend(["--companion", str(models["companion"])])
    if "transcriber" in models and models["transcriber"].exists():
        cmd.extend(["--transcriber", str(models["transcriber"])])
    if adapters_dir and adapters_dir.exists():
        has_adapters = any(p.is_dir() for p in adapters_dir.iterdir())
        if has_adapters:
            cmd.extend(["--adapters", str(adapters_dir)])
            logger.info("Configured LoRA adapters directory: %s", adapters_dir)

    logger.info("Executing yue-server on GPU %d (CUDA%d): %s", gpu_id, gpu_id, " ".join(cmd))
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    env["GGML_BACKEND"] = "CUDA0"

    work_dir = executable.parent
    build_dir = work_dir / "yue2.cpp" / "build"
    env["LD_LIBRARY_PATH"] = f"{work_dir}:{build_dir}:{env.get('LD_LIBRARY_PATH', '')}"

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    return process


def ensure_muscriptor_model(work_dir: Path, size: str = "large") -> Path:
    """Ensures MuScriptor weights (small/medium/large) are downloaded in work_dir."""
    size_norm = size.lower().strip()
    if "small" in size_norm or "103" in size_norm:
        key = "small"
    elif "medium" in size_norm or "307" in size_norm:
        key = "medium"
    else:
        key = "large"

    target_dir = work_dir / "models" / f"muscriptor-{key}"
    target_dir.mkdir(parents=True, exist_ok=True)
    safetensors = target_dir / "model.safetensors"
    config_json = target_dir / "config.json"

    specs = {
        "large": (
            "https://huggingface.co/cocktailpeanut/muscriptor-large/resolve/87f4bf981f56f90fb5043153b3f54af3c3053da9/model.safetensors",
            "https://huggingface.co/cocktailpeanut/muscriptor-large/resolve/87f4bf981f56f90fb5043153b3f54af3c3053da9/config.json",
            5_000_000_000,
        ),
        "medium": (
            "https://huggingface.co/cocktailpeanut/muscriptor-medium/resolve/27246ba68bd4d8f98bdec10a6edf8d7cf42a8826/model.safetensors",
            "https://huggingface.co/cocktailpeanut/muscriptor-medium/resolve/27246ba68bd4d8f98bdec10a6edf8d7cf42a8826/config.json",
            1_000_000_000,
        ),
        "small": (
            "https://huggingface.co/cocktailpeanut/muscriptor-small/resolve/31a8f75d6a8b5383fd71ad1371dc1620389ab722/model.safetensors",
            "https://huggingface.co/cocktailpeanut/muscriptor-small/resolve/31a8f75d6a8b5383fd71ad1371dc1620389ab722/config.json",
            300_000_000,
        ),
    }

    url_st, url_cfg, min_bytes = specs[key]

    if not safetensors.exists() or safetensors.stat().st_size < min_bytes:
        logger.info("Downloading MuScriptor %s weights into %s...", key, target_dir)
        has_aria2 = shutil.which("aria2c") is not None
        if has_aria2:
            subprocess.check_call(
                [
                    "aria2c",
                    "-c",
                    "-x",
                    "16",
                    "-s",
                    "16",
                    "-k",
                    "1M",
                    "--file-allocation=none",
                    "--dir",
                    str(target_dir),
                    "--out",
                    "model.safetensors",
                    url_st,
                ]
            )
            subprocess.check_call(
                [
                    "aria2c",
                    "-c",
                    "--dir",
                    str(target_dir),
                    "--out",
                    "config.json",
                    url_cfg,
                ]
            )
        else:
            urllib.request.urlretrieve(url_st, str(safetensors))
            urllib.request.urlretrieve(url_cfg, str(config_json))
        logger.info("MuScriptor %s weights ready at %s", key, safetensors)

    return safetensors


class UnifiedServerHandler(http.server.BaseHTTPRequestHandler):
    """Unified HTTP handler: proxies generation to yue-server, handles /transcribe via MuScriptor on GPU 1."""

    backend_port: int = 18088
    work_dir: Path = Path("/kaggle/working")
    transcribe_gpu_id: int = 1

    def do_GET(self) -> None:
        self._proxy("GET")

    def do_POST(self) -> None:
        if self.path.startswith("/transcribe"):
            self._handle_transcribe()
            return
        self._proxy("POST")

    def do_HEAD(self) -> None:
        self._proxy("HEAD")

    def _proxy(self, method: str) -> None:
        target = f"http://127.0.0.1:{self.backend_port}{self.path}"
        content_length = int(self.headers.get("Content-Length", 0))
        body = self.rfile.read(content_length) if content_length > 0 else None

        req_headers: dict[str, str] = {}
        for header, value in self.headers.items():
            if header.lower() not in ("host", "content-length"):
                req_headers[header] = value

        req = urllib.request.Request(
            target,
            data=body,
            headers=req_headers,
            method=method,
        )
        try:
            with urllib.request.urlopen(req, timeout=600) as resp:
                self.send_response(resp.status)
                for header, value in resp.headers.items():
                    if header.lower() not in ("transfer-encoding", "content-length"):
                        self.send_header(header, value)
                resp_data = resp.read()
                self.send_header("Content-Length", str(len(resp_data)))
                self.end_headers()
                self.wfile.write(resp_data)
        except urllib.error.HTTPError as exc:
            self.send_response(exc.code)
            self.end_headers()
            self.wfile.write(exc.read())
        except Exception as exc:  # noqa: BLE001
            logger.error("Internal proxy error: %s", exc)
            self.send_response(502)
            self.end_headers()
            self.wfile.write(str(exc).encode())

    def _handle_transcribe(self) -> None:
        try:
            content_length = int(self.headers.get("Content-Length", 0))
            if content_length <= 0:
                self.send_response(400)
                self.end_headers()
                self.wfile.write(b"Empty audio payload")
                return

            model_size = self.headers.get("X-Model-Size", "large").strip().lower()
            audio_data = self.rfile.read(content_length)
            tmp_in = Path("/tmp/input_transcribe.mp3")
            tmp_out = Path("/tmp/output_transcribe.mid")
            tmp_in.write_bytes(audio_data)

            if tmp_out.exists():
                tmp_out.unlink()

            if shutil.which("muscriptor") is None:
                logger.info("Installing muscriptor package...")
                subprocess.check_call(
                    [sys.executable, "-m", "pip", "install", "-q", "muscriptor"]
                )

            model_path = ensure_muscriptor_model(self.work_dir, size=model_size)

            logger.info(
                "Transcribing audio (%d bytes, model=%s) via MuScriptor on GPU %d...",
                len(audio_data),
                model_size,
                self.transcribe_gpu_id,
            )

            cmd = [
                "muscriptor",
                "transcribe",
                str(tmp_in),
                "-o",
                str(tmp_out),
                "--model",
                str(model_path),
                "--device",
                "cuda",
                "--dtype",
                "float16",
            ]
            env = os.environ.copy()
            env["CUDA_VISIBLE_DEVICES"] = str(self.transcribe_gpu_id)

            res = subprocess.run(cmd, capture_output=True, text=True, env=env)
            if res.returncode != 0:
                err_text = (res.stderr or res.stdout or "").strip()
                logger.warning(
                    "MuScriptor GPU %d attempt failed (code %d): %s",
                    self.transcribe_gpu_id,
                    res.returncode,
                    err_text,
                )
                logger.info("Retrying MuScriptor on CPU...")
                cmd_cpu = [
                    "muscriptor",
                    "transcribe",
                    str(tmp_in),
                    "-o",
                    str(tmp_out),
                    "--model",
                    str(model_path),
                    "--device",
                    "cpu",
                    "--dtype",
                    "float32",
                ]
                res = subprocess.run(cmd_cpu, capture_output=True, text=True)

            if not tmp_out.exists():
                err = res.stderr or res.stdout or "Transcription produced no output"
                logger.error("Transcription failed: %s", err)
                self.send_response(500)
                self.end_headers()
                self.wfile.write(f"Transcription failed: {err}".encode())
                return

            midi_bytes = tmp_out.read_bytes()
            logger.info("Transcription succeeded (%d bytes MIDI produced)", len(midi_bytes))
            self.send_response(200)
            self.send_header("Content-Type", "audio/midi")
            self.send_header("Content-Length", str(len(midi_bytes)))
            self.end_headers()
            self.wfile.write(midi_bytes)

            if tmp_in.exists():
                tmp_in.unlink()
            if tmp_out.exists():
                tmp_out.unlink()
        except Exception as exc:  # noqa: BLE001
            logger.error("Transcribe error: %s", exc)
            self.send_response(500)
            self.end_headers()
            self.wfile.write(f"Transcribe error: {exc}".encode())

    def log_message(self, format: str, *args: object) -> None:
        if args and str(args[0]).startswith("GET /health"):
            return
        logger.info("%s - %s", self.address_string(), format % args)


class ReusableServer(http.server.ThreadingHTTPServer):
    """Threading HTTPServer with socket SO_REUSEADDR enabled."""

    allow_reuse_address = True


def print_banner(tunnel_url: str, gpu_count: int) -> None:
    """Prints user instructions with clickable banner and HTML card."""
    line = "=" * 70
    msg = (
        f"\n{line}\n"
        f"🚀 YuE2 STUDIO KAGGLE ENGINE (2x TESLA T4) IS READY!\n"
        f"{line}\n"
        f"🎮 Аппаратное распределение:\n"
        f"   • GPU 0 (Tesla T4 #1, 16 GB): Музыкальная генерация YuE2 (Q8_0 / BF16)\n"
        f"   • GPU 1 (Tesla T4 #2, 16 GB): Транскрибация Аудио в MIDI (MuScriptor Large)\n"
        f"{line}\n"
        f"🔗 ССЫЛКА ДЛЯ connect-colab.bat:\n"
        f"   >>> {tunnel_url} <<<\n"
        f"{line}\n"
        f"📋 Инструкция:\n"
        f"  1. Скопируй ссылку выше: {tunnel_url}\n"
        f"  2. Запусти на ПК файл connect-colab.bat и вставь эту ссылку\n"
        f"     (или укажи в YuE2 Studio: Настройки -> Движок -> Внешний сервер)\n"
        f"  3. Нажми 'Проверить' -> 'Сохранить и применить'\n"
        f"  4. Наслаждайся параллельной работой двух GPU без риска нехватки VRAM!\n"
        f"{line}\n"
    )
    print(msg, flush=True)


def notify_ready(tunnel_url: str, topic: str | None = None) -> bool:
    """Publishes the tunnel URL to ntfy.sh for instant browserless desktop discovery."""
    import hashlib
    import urllib.request

    if not topic:
        user = os.environ.get("KAGGLE_USERNAME", "alexeyklochkov")
        h = hashlib.sha256(user.strip().lower().encode("utf-8")).hexdigest()[:16]
        topic = f"yue2_studio_{h}"

    ntfy_url = f"https://ntfy.sh/{topic}"
    try:
        req = urllib.request.Request(
            ntfy_url,
            data=tunnel_url.encode("utf-8"),
            headers={
                "Title": "YuE2 Server Ready (2x Tesla T4)",
                "Tags": "musical_note,rocket",
                "Priority": "high",
            },
        )
        with urllib.request.urlopen(req, timeout=8) as resp:
            if resp.status == 200:
                logger.info("📡 Published tunnel URL to ntfy.sh topic: %s", topic)
                return True
    except Exception as exc:
        logger.warning("Could not publish to ntfy.sh: %s", exc)
    return False


def run_kaggle_server(
    model: str = "q8_0",
    port: int = DEFAULT_PORT,
    tunnel: str = "cloudflare",
    ngrok_token: str = "",
    keep_loaded: bool = True,
    no_transcriber: bool = False,
) -> None:
    """Initializes and runs the dual-GPU server on Kaggle."""
    work_dir, gpu_count = detect_environment()
    work_dir.mkdir(parents=True, exist_ok=True)
    models_dir = work_dir / "models"
    models_dir.mkdir(parents=True, exist_ok=True)

    adapters_dir = work_dir / "adapters"
    adapters_dir.mkdir(parents=True, exist_ok=True)

    # Determine GPU roles
    gen_gpu_id = 0
    transcribe_gpu_id = 1 if gpu_count >= 2 else 0

    backend_port = port + 1

    # Cleanup stale processes
    try:
        subprocess.call(["killall", "-9", "yue-server"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        subprocess.call(["killall", "-9", "cloudflared"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(1)
    except Exception:
        pass

    ensure_system_packages()
    engine_exe = build_yue_engine(work_dir)
    models = download_models(models_dir, model.lower(), include_transcriber=not no_transcriber)

    # Launch yue-server on GPU 0
    engine_proc = launch_engine(
        engine_exe,
        models,
        adapters_dir=adapters_dir,
        port=backend_port,
        gpu_id=gen_gpu_id,
    )

    # Start Unified HTTP Server on primary port
    UnifiedServerHandler.backend_port = backend_port
    UnifiedServerHandler.work_dir = work_dir
    UnifiedServerHandler.transcribe_gpu_id = transcribe_gpu_id

    httpd = ReusableServer(("0.0.0.0", port), UnifiedServerHandler)
    server_thread = threading.Thread(target=httpd.serve_forever, daemon=True)
    server_thread.start()

    tunnel_proc: subprocess.Popen[str] | None = None
    tunnel_url = f"http://127.0.0.1:{port}"

    if tunnel == "cloudflare":
        tunnel_proc, tunnel_url = setup_cloudflare_tunnel(port, work_dir)
    elif tunnel == "ngrok":
        tunnel_proc, tunnel_url = setup_ngrok_tunnel(port, ngrok_token or None)

    print_banner(tunnel_url, gpu_count)
    notify_ready(tunnel_url)

    banner_shown = False
    try:
        while True:
            if engine_proc.stdout:
                line = engine_proc.stdout.readline()
                if line:
                    sys.stdout.write(line)
                    sys.stdout.flush()
                    if not banner_shown and "listening on" in line.lower():
                        print_banner(tunnel_url, gpu_count)
                        banner_shown = True
            if engine_proc.poll() is not None:
                logger.error("Engine process terminated with code %d", engine_proc.returncode)
                break
            time.sleep(0.01)
    except KeyboardInterrupt:
        logger.info("Stopping engine and tunnel...")
    finally:
        httpd.server_close()
        engine_proc.terminate()
        if tunnel_proc:
            tunnel_proc.terminate()
        try:
            from pyngrok import ngrok  # type: ignore[import-untyped]
            ngrok.kill()
        except Exception:
            pass


def main() -> None:
    parser = argparse.ArgumentParser(description="YuE2 Studio Kaggle Dual-GPU Server")
    parser.add_argument("--model", default="q8_0", choices=["q8_0", "bf16", "q6_k", "q5_k_m"])
    parser.add_argument("--port", type=int, default=DEFAULT_PORT)
    parser.add_argument("--tunnel", default="cloudflare", choices=["cloudflare", "ngrok"])
    parser.add_argument("--ngrok-token", default="")
    parser.add_argument("--keep-loaded", action="store_true", default=True)
    parser.add_argument("--no-transcriber", action="store_true", default=False)
    args = parser.parse_args()

    run_kaggle_server(
        model=args.model,
        port=args.port,
        tunnel=args.tunnel,
        ngrok_token=args.ngrok_token,
        keep_loaded=args.keep_loaded,
        no_transcriber=args.no_transcriber,
    )



# Запуск двухпроцессорного сервера на Kaggle (GPU 0 + GPU 1)
run_kaggle_server(
    model=QUANT.lower(),
    port=PORT,
    tunnel=TUNNEL,
    ngrok_token=NGROK_TOKEN,
)


### 🎨 Шаг 2.5: Скачивание LoRA стилей (опционально)
Выберите стиль из каталога или вставьте Hugging Face ссылку. Стили сохраняются в `/kaggle/working/adapters` и автоматически подхватываются сервером.

In [ ]:
# @title 🎨 Скачать LoRA стили { run: "auto" }
LORA_PRESET = "yue2-epic-trailer"  # @param ["yue2-epic-trailer", "yue2-industrial-rock", "yue2-dreampop", "yue2-oldschool-hiphop", "yue2-militant-reggae", "yue2-instrumental", "yue2-sv-billie", "yue2-canzone-italiana", "yue2-chanson-francaise", "ALL_POPULAR", "CUSTOM_URL"]
CUSTOM_URL = ""  # @param {type:"string"}
CUSTOM_NAME = ""  # @param {type:"string"}  

import os
import shutil
import subprocess
import urllib.request
from pathlib import Path

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
ADAPTERS_DIR = WORK_DIR / "adapters"
ADAPTERS_DIR.mkdir(parents=True, exist_ok=True)

LORA_CATALOG = {
    "yue2-epic-trailer": [
        ("https://huggingface.co/monsterovich/yue2-steps-from-hell/resolve/bc84ad1867158e1112ab9601c74b338847d9dff2/adapter-ar-195/lora.safetensors", "epic-ar.safetensors"),
        ("https://huggingface.co/monsterovich/yue2-steps-from-hell/resolve/bc84ad1867158e1112ab9601c74b338847d9dff2/adapter-nar-194/lora.safetensors", "epic-nar.safetensors"),
    ],
    "yue2-industrial-rock": [
        ("https://huggingface.co/monsterovich/yue2-industrial-rock-lora/resolve/a71706d59e2874f1e6db9aa10e8da5cc19e6c924/adapter-ar-179/lora.safetensors", "industrial-ar.safetensors"),
        ("https://huggingface.co/monsterovich/yue2-industrial-rock-lora/resolve/a71706d59e2874f1e6db9aa10e8da5cc19e6c924/adapter-nar-179-v2/lora.safetensors", "industrial-nar.safetensors"),
    ],
    "yue2-dreampop": [
        ("https://huggingface.co/atomtanstudio/lora-library/resolve/b5727622224c267030be0054464d953188edba49/yue2/dreampop/dreampop_sv_dreampop.safetensors", "dreampop_sv_dreampop.safetensors"),
    ],
    "yue2-oldschool-hiphop": [
        ("https://huggingface.co/atomtanstudio/lora-library/resolve/b5727622224c267030be0054464d953188edba49/yue2/oldschoolhiphop/sv_oldschoolhiphop.safetensors", "sv_oldschoolhiphop.safetensors"),
    ],
    "yue2-militant-reggae": [
        ("https://huggingface.co/becausereasons/yue2-mltnt-militant-reggae/resolve/a9fedc857cd1f3952be3177326c76526abcea53a/mltnt_steppers.safetensors", "mltnt_steppers.safetensors"),
    ],
    "yue2-instrumental": [
        ("https://huggingface.co/Mothersuperior/YuE2-instrumental-cot-full-loras/resolve/947f2f4b28978b2b6c3e316e6a87925c76bf3c4b/ar_lora_inst_v3abc.bf16.safetensors", "ar_lora_inst_v3abc.bf16.safetensors"),
    ],
    "yue2-sv-billie": [
        ("https://huggingface.co/HaileyStorm/sv-billie-yue2-lora/resolve/94fe9fbf5dfd3e97eefeb3ef54e2801eb3b69077/sv_billie.safetensors", "sv_billie.safetensors"),
    ],
    "yue2-canzone-italiana": [
        ("https://huggingface.co/becausereasons/yue2-cnzn-canzone-italiana/resolve/6755490a2a50aa738f6ee2838384ec6efdb3a13f/cnzn_sanremo.safetensors", "cnzn_sanremo.safetensors"),
    ],
    "yue2-chanson-francaise": [
        ("https://huggingface.co/becausereasons/yue2-chnsn-chanson-francaise/resolve/d12d09b6716075196f7c10b7b134d19aa786c23c/chnsn_montmartre.safetensors", "chnsn_montmartre.safetensors"),
    ],
}

def download_files(adapter_id: str, files: list[tuple[str, str]]) -> None:
    dest_dir = ADAPTERS_DIR / adapter_id
    dest_dir.mkdir(parents=True, exist_ok=True)
    for url, filename in files:
        target = dest_dir / filename
        if target.exists() and target.stat().st_size > 1_000_000:
            print(f"  ✓ {filename} уже скачан.")
            continue
        print(f"  ⏳ Скачивание {filename}...")
        if shutil.which("aria2c"):
            subprocess.run([
                "aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M",
                "--file-allocation=none", url, "-d", str(dest_dir), "-o", filename
            ], check=True)
        else:
            urllib.request.urlretrieve(url, str(target))
        print(f"  ✅ {filename} сохранён!")

if LORA_PRESET == "ALL_POPULAR":
    for name, flist in LORA_CATALOG.items():
        print(f"\n📦 Скачивание LoRA '{name}'...")
        download_files(name, flist)
elif LORA_PRESET == "CUSTOM_URL":
    if not CUSTOM_URL:
        print("❌ Укажите ссылку CUSTOM_URL")
    else:
        name = CUSTOM_NAME.strip() or "custom-lora"
        fname = Path(CUSTOM_URL.split("?")[0]).name or "lora.safetensors"
        print(f"\n📦 Скачивание кастомной LoRA '{name}'...")
        download_files(name, [(CUSTOM_URL, fname)])
elif LORA_PRESET in LORA_CATALOG:
    print(f"\n📦 Скачивание LoRA '{LORA_PRESET}'...")
    download_files(LORA_PRESET, LORA_CATALOG[LORA_PRESET])

print("\n" + "=" * 65)
print(f"📂 Установленные LoRA ({ADAPTERS_DIR}):")
installed = [p.name for p in ADAPTERS_DIR.iterdir() if p.is_dir()]
if installed:
    for item in sorted(installed):
        count = len(list((ADAPTERS_DIR / item).glob("*.safetensors")))
        print(f"  • {item} ({count} safetensors)")
else:
    print("  (пока пусто)")
print("=" * 65)


### 🔍 Шаг 3: Диагностика и статус видеокарт
Проверяет состояние обеих видеокарт (память VRAM на GPU 0 и GPU 1), файлы моделей и готовность API.

In [ ]:
# @title 🔍 Диагностика 2x T4 и статуса сервера
!nvidia-smi --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv

import urllib.request
import json
from pathlib import Path

work_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
models_dir = work_dir / "models"

print("\n📂 Файлы моделей в", models_dir)
if models_dir.exists():
    for f in sorted(models_dir.rglob("*")):
        if f.is_file():
            size_mb = f.stat().st_size / (1024 * 1024)
            print(f"  • {f.name} ({size_mb:.1f} MB)")
else:
    print("  (модели еще не скачаны)")

print("\n🌐 Проверка локального API (порт 18087):")
try:
    with urllib.request.urlopen("http://127.0.0.1:18087/props", timeout=3) as resp:
        data = json.loads(resp.read())
        print("  ✅ Движок yue-server отвечает! Версия:", data.get("version", "ok"))
except Exception as exc:
    print("  ℹ️ Сервер еще запускается или не запущен:", exc)


### 🎹 Шаг 4: Прямая транскрибация Аудио в MIDI (MuScriptor на GPU 1)
Если хотите протестировать конвертацию песни в ноты/MIDI прямо в блокноте (на выделенной GPU 1 с моделью Large).

In [ ]:
# @title 🎹 Транскрибация Аудио в MIDI (MuScriptor на GPU 1) { run: "auto" }
AUDIO_PATH = ""  # @param {type:"string"}
MODEL_CHOICE = "large"  # @param ["large", "medium", "small"]

import os
import sys
import shutil
import subprocess
from pathlib import Path

if shutil.which("muscriptor") is None:
    print("📦 Установка muscriptor...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "muscriptor"])

work_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
model_dir = work_dir / "models" / f"muscriptor-{MODEL_CHOICE}"
model_dir.mkdir(parents=True, exist_ok=True)
safe_tensors = model_dir / "model.safetensors"
config_json = model_dir / "config.json"

URLS = {
    "large": ("https://huggingface.co/cocktailpeanut/muscriptor-large/resolve/87f4bf981f56f90fb5043153b3f54af3c3053da9/model.safetensors", "https://huggingface.co/cocktailpeanut/muscriptor-large/resolve/87f4bf981f56f90fb5043153b3f54af3c3053da9/config.json", 5_000_000_000),
    "medium": ("https://huggingface.co/cocktailpeanut/muscriptor-medium/resolve/27246ba68bd4d8f98bdec10a6edf8d7cf42a8826/model.safetensors", "https://huggingface.co/cocktailpeanut/muscriptor-medium/resolve/27246ba68bd4d8f98bdec10a6edf8d7cf42a8826/config.json", 1_000_000_000),
    "small": ("https://huggingface.co/cocktailpeanut/muscriptor-small/resolve/31a8f75d6a8b5383fd71ad1371dc1620389ab722/model.safetensors", "https://huggingface.co/cocktailpeanut/muscriptor-small/resolve/31a8f75d6a8b5383fd71ad1371dc1620389ab722/config.json", 300_000_000),
}
url_st, url_cfg, min_bytes = URLS[MODEL_CHOICE]
if not safe_tensors.exists() or safe_tensors.stat().st_size < min_bytes:
    print(f"⏳ Скачивание MuScriptor {MODEL_CHOICE}...")
    if shutil.which("aria2c"):
        subprocess.check_call(["aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M", "--file-allocation=none", url_st, "-d", str(model_dir), "-o", "model.safetensors"])
        subprocess.check_call(["aria2c", "-c", url_cfg, "-d", str(model_dir), "-o", "config.json"])
    else:
        import urllib.request
        urllib.request.urlretrieve(url_st, str(safe_tensors))
        urllib.request.urlretrieve(url_cfg, str(config_json))

if not AUDIO_PATH or not Path(AUDIO_PATH).exists():
    print("ℹ️ Укажите путь к аудиофайлу в AUDIO_PATH (например, /kaggle/working/track.mp3).")
    print("   (Напоминание: YuE2 Studio умеет вызывать эту транскрибацию автоматически через кнопку волшебной палочки)")
else:
    out_midi = f"{Path(AUDIO_PATH).stem}.mid"
    print(f"⚡ Транскрибация {AUDIO_PATH} на GPU 1 (Tesla T4) через MuScriptor {MODEL_CHOICE}...")
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "1"
    cmd = ["muscriptor", "transcribe", str(AUDIO_PATH), "-o", str(out_midi), "--model", str(safe_tensors), "--device", "cuda", "--dtype", "float16"]
    subprocess.run(cmd, check=True, env=env)
    print(f"🎉 Готово! Создан файл MIDI: {out_midi}")
